# Build Annotation Files

This notebook builds the **WordNet-based candidate lexicon** for the entity schema, scores every sentence in the corpus for nature-entity density, and draws a sample for **manual annotation**.

**Prerequisite:** run `setup.ipynb` first (needs `data/corpus/*.txt` and `data/corpus/corpus_metadata.csv`).

**Entity schema reference**:

| Category | Covers |
|---|---|
| FLORA | trees, flowers, moss, woods |
| FAUNA | animals, birds |
| WEATHER | wind, storm, rain, cloud, sun, moon, stars |
| LANDSCAPE | rivers, mountains, glaciers, named places |
| NATURE | personified nature-as-force - abstract address or a character (e.g. the Witch of the Alps) |

In [1]:
import os
import re
import csv
import random

import spacy
from nltk.corpus import wordnet as wn

In [ ]:
NOTEBOOK_DIR = os.path.abspath(os.getcwd())

if os.path.basename(NOTEBOOK_DIR) == "notebooks":
    PROJECT_ROOT = os.path.dirname(NOTEBOOK_DIR)
else:
    PROJECT_ROOT = NOTEBOOK_DIR

CORPUS_DIR = os.path.join(PROJECT_ROOT, "data", "corpus")
METADATA_PATH = os.path.join(CORPUS_DIR, "corpus_metadata.csv")
SENTENCE_SCORES_PATH = os.path.join(PROJECT_ROOT, "data", "annotations", "sentence_scores.csv")
ANNOTATION_POOL_PATH = os.path.join(PROJECT_ROOT, "data", "annotations", "annotation_pool.csv")

RANDOM_SEED = 42

# WordNet synset roots to expand into candidate word lists per entity type.
ENTITY_SYNSETS = {
    "FLORA": ["plant.n.02", "tree.n.01", "flower.n.01", "wood.n.01"],
    "FAUNA": ["animal.n.01", "bird.n.01"],
    "WEATHER": ["atmospheric_phenomenon.n.01", "weather.n.01", "wind.n.01",
               "cloud.n.02", "storm.n.01", "celestial_body.n.01"],
    "LANDSCAPE": ["geological_formation.n.01", "body_of_water.n.01",
                  "mountain.n.01", "valley.n.01"],
}

# Density bands: (min_candidates, max_candidates_inclusive_or_None)
BAND_EDGES = [
    ("band_0", 0, 0),
    ("band_1", 1, 1),
    ("band_2", 2, 2),
    ("band_3plus", 3, None),
]

# What fraction of the total sample should come from each band.
# Skews toward richer sentences, still keeps some "empty" sentences so the NER model learns what a non-entity looks like.
BAND_QUOTAS = {
    "band_0": 0.10,
    "band_1": 0.30,
    "band_2": 0.30,
    "band_3plus": 0.30,
}

TOTAL_SAMPLE_SIZE = 350

# Safety switch: if annotation_pool.csv already exists, this run will refuse to overwrite it rather than silently rewrite the work.
# Set to True only when you deliberately want to regenerate the pool from scratch.
FORCE_OVERWRITE = False

NATURE_KEYWORD_PATTERN = re.compile(r"\bNature\b")
NATURE_BOOST_CAP = 40

## Lexicon Builder

Expands the entity-schema synset roots into candidate word lists, with a curated exclusion list for known false-positive senses.

In [ ]:
# Manual exclusions for WordNet false positives discovered during testing:
# 1. Human terms (WordNet maps humans under 'animal.n.01', which breaks our FAUNA tags).
# 2. High-frequency generic words with rare nature senses (prevents massive over-matching).
EXCLUDE_WORDS = {
    "man", "men", "mankind", "woman", "women", "human", "humans",
    "person", "people", "mortal", "mortals", "folk", "world",
    "head", "side", "sound", "breath",
}


def build_lexicon():
    lexicon = {}  # entity_type -> set of lowercase words (spaces -> underscores removed)
    for entity_type, synset_names in ENTITY_SYNSETS.items():
        words = set()
        for synset_name in synset_names:
            try:
                root = wn.synset(synset_name)
            except Exception:
                print(f" Warning: synset '{synset_name}' not found, skipping.")
                continue
            # root itself + every hyponym (descendant) recursively
            all_synsets = set([root])
            all_synsets.update(root.closure(lambda s: s.hyponyms()))
            for syn in all_synsets:
                for lemma in syn.lemma_names():
                    word = lemma.replace("_", " ").lower()
                    # skip overly generic / multi-word entries that would over-match (e.g. single-letter or very long compounds)
                    if 2 <= len(word) <= 30:
                        words.add(word)

        removed = words & EXCLUDE_WORDS
        words -= EXCLUDE_WORDS                
        lexicon[entity_type] = words

        print(f"{entity_type}: {len(words)} candidate words "
              f"({len(removed)} excluded)")
    return lexicon

# Take the five categories and merge them into one dictionary for fast lookup during the actual sentence-scoring pass
def flatten_lexicon(lexicon):
    combined = {}
    for entity_type, words in lexicon.items():
        for w in words:
            combined.setdefault(w, set()).add(entity_type)
    return combined

## Sentence Scoring

Scores each sentence by how many entity matches it contains against the WordNet-derived lexicon, then maps that score into a density band for stratified sampling.

- `score_sentence()`: checks each NOUN/PROPN token's lemma (and adjacent NOUN/PROPN pairs, for multi-word WordNet entries like "oak tree") against the lexicon. spaCy supplies the POS tagging and lemmatization this depends on.
- `band_for_score()`: converts the resulting match count into one of the density bands defined in `BAND_EDGES`, used later for stratified sampling across band_0/1/2/3+.

In [4]:
def score_sentence(spacy_sent, combined_lexicon):
    tokens = [t for t in spacy_sent if t.is_alpha]  # this keeps only alphabetic tokens —> strips out punctuation, numbers, and whitespace
    matched_words = []
    matched_types = set()

    # single-word matches (lemma, noun/proper-noun only)
    for tok in tokens:
        if tok.pos_ not in ("NOUN", "PROPN"):
            continue
        lemma = tok.lemma_.lower()
        if lemma in combined_lexicon:
            matched_words.append(lemma)
            matched_types.update(combined_lexicon[lemma])

    # two-word phrase matches (for multi-word WordNet lemmas, e.g. "oak tree")
    for i in range(len(tokens) - 1):
        if tokens[i + 1].pos_ not in ("NOUN", "PROPN"):
            continue
        phrase = f"{tokens[i].lemma_.lower()} {tokens[i+1].lemma_.lower()}"
        if phrase in combined_lexicon:
            matched_words.append(phrase)
            matched_types.update(combined_lexicon[phrase])

    return len(matched_words), matched_words, matched_types


def band_for_score(score):
    for band_name, lo, hi in BAND_EDGES:
        if hi is None:
            if score >= lo:
                return band_name
        elif lo <= score <= hi:
            return band_name
    return BAND_EDGES[-1][0]

## Load Corpus

Loads author/work metadata from the corpus_metadata built in `setup.ipynb`.

In [5]:
def load_corpus_metadata():
    if not os.path.exists(METADATA_PATH):
        raise FileNotFoundError(
            f"{METADATA_PATH} not found -- run setup.ipynb first."
        )
    meta = {}
    with open(METADATA_PATH, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            meta[row["output_file"]] = (row["author"], row["work"])
    return meta

## Run

Score every sentence in the corpus against the lexicon, write the full scored corpus to `sentence_scores.csv`, then stratified-samples across density bands into `annotation_pool.csv`, the template ready for manual annotation.

In [6]:
def main():
    if os.path.exists(ANNOTATION_POOL_PATH) and not FORCE_OVERWRITE:
        print(
            f"SKIPPED: annotation_pool.csv already exists.\n\n"
            f"If you genuinely want to regenerate the pool from scratch, set FORCE_OVERWRITE = True at the top of the notebook."
        )
        return

    random.seed(RANDOM_SEED)

    # ensure output directories exist before anything tries to write to them
    for path in (SENTENCE_SCORES_PATH, ANNOTATION_POOL_PATH):
        out_dir = os.path.dirname(path)
        if out_dir:
            os.makedirs(out_dir, exist_ok=True)

    nlp = spacy.load("en_core_web_sm", disable=["ner"])         # speed optimization -> skip spaCy's own named-entity pipeline component

    lexicon = build_lexicon()
    combined_lexicon = flatten_lexicon(lexicon)
    print(f"Total unique candidate words: {len(combined_lexicon)}")

    corpus_metadata_meta = load_corpus_metadata()

    if not os.path.isdir(CORPUS_DIR):
        print(f"ERROR: '{CORPUS_DIR}' directory not found. ")
        return

    all_rows = []
    sentence_id_counter = 0

    corpus_files = sorted(
        f for f in os.listdir(CORPUS_DIR) if f.endswith(".txt")
    )
    if not corpus_files:
        print(f"ERROR: no .txt files found in '{CORPUS_DIR}'.")
        return

    for filename in corpus_files:
        filepath = os.path.join(CORPUS_DIR, filename)
        rel_path = filepath  # matches corpus_metadata's output_file format
        if rel_path in corpus_metadata_meta:
            author, work = corpus_metadata_meta[rel_path]
        else:
            raise KeyError(
                f"File not found - re-run setup.ipynb to regenerate it."
            )

        with open(filepath, "r", encoding="utf-8", errors="replace") as f:
            text = f.read()

        # Normalise whitespace/line-breaks before sentence splitting, so poetry line breaks don't get treated as sentence boundaries -> collapsing whitespace first lets the grammar, not the typesetting, decide where sentences end.
        text_for_parsing = re.sub(r"\s+", " ", text).strip()

        # spaCy's statistical sentence segmenter
        doc = nlp(text_for_parsing)
        for sent in doc.sents:
            sent_text = sent.text.strip()
            if len(sent_text.split()) < 3:      # Anything under 3 words gets discarded
                continue
            score, matched_words, matched_types = score_sentence(
                sent, combined_lexicon
            )
            band = band_for_score(score)
            sentence_id_counter += 1
            all_rows.append({
                "sentence_id": f"s{sentence_id_counter:05d}",
                "author": author,
                "work": work,
                "sentence_text": sent_text,
                "n_candidates": score,
                "candidate_words": "; ".join(sorted(set(matched_words))),
                "candidate_entity_types": "; ".join(sorted(matched_types)),
                "band": band,
            })

    print(f"Total sentences parsed: {len(all_rows)}")

    with open(SENTENCE_SCORES_PATH, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(all_rows[0].keys()))
        writer.writeheader()
        writer.writerows(all_rows)
    print(f"Wrote sentence_scores.csv")

    # band summary
    band_counts = {}
    for row in all_rows:
        band_counts.setdefault(row["band"], []).append(row)

    # stratified sampling based on band quotas
    sampled = []
    for band_name, _, _ in BAND_EDGES:
        quota = int(round(TOTAL_SAMPLE_SIZE * BAND_QUOTAS[band_name]))
        available = band_counts.get(band_name, [])
        take = min(quota, len(available))
        
        if take < quota:
            print(f"WARNING: {band_name} shortfall (wanted {quota}, got {len(available)})")
            
        chosen = random.sample(available, take) if available else []
        sampled.extend(chosen)

    print(f"Sampled {len(sampled)} total sentences for annotation pool.")

    # Include sentences with capitalized "Nature" missed by WordNet
    already_ids = {row["sentence_id"] for row in sampled}
    nature_hits = [
        row for row in all_rows
        if row["sentence_id"] not in already_ids
        and NATURE_KEYWORD_PATTERN.search(row["sentence_text"])
    ]
    nature_added = nature_hits[:NATURE_BOOST_CAP]
    sampled.extend(nature_added)
    
    print(f"Nature boost: added {len(nature_added)} sentences.")
    print(f"Final total sampled for annotation: {len(sampled)}")

    # Write annotation template with blank target columns
    annotation_fields = [
        "sentence_id", "author", "work", "sentence_text",
        "band", "n_candidates", "candidate_words",
        "target_span", "start_char", "end_char", "entity_type",
        "is_figurative", "lemma", "notes",
    ]
    
    with open(ANNOTATION_POOL_PATH, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=annotation_fields)
        writer.writeheader()
        for row in sampled:
            writer.writerow({
                "sentence_id": row["sentence_id"],
                "author": row["author"],
                "work": row["work"],
                "sentence_text": row["sentence_text"],
                "band": row["band"],
                "n_candidates": row["n_candidates"],
                "candidate_words": row["candidate_words"],
                "target_span": "",
                "start_char": "",
                "end_char": "",
                "entity_type": "",
                "is_figurative": "",
                "lemma": "",
                "notes": "",
            })
            
    print(f"\nWrote annotation_pool.csv - ready for manual annotation.")
    print("See docs/annotation_guideline.md for instructions on filling out the target columns.")


main()

SKIPPED: annotation_pool.csv already exists.

If you genuinely want to regenerate the pool from scratch, set FORCE_OVERWRITE = True at the top of the notebook.
